# MeMo Step B — Memory Model SFT on Google Colab

This notebook fine-tunes a small Memory Model (`Qwen2.5-1.5B-Instruct` by default) on the
reflection Q&A dataset produced by Step A. Designed for a free-tier Colab T4 GPU.

**Before running**: switch the Colab runtime to GPU
(`Runtime` → `Change runtime type` → `T4 GPU`).

## 1. Verify GPU

In [1]:
!nvidia-smi

Sun May 31 10:54:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the project into Colab

Pick **ONE** of the two cells below.

**Option A** — if you've pushed this repo to GitHub:

In [ ]:
# !git clone https://github.com/<your-user>/<your-repo>.git memo_project
# %cd memo_project

In [3]:
import os
print(os.listdir())

['.config', 'sample_data']


**Option B** — upload a zip of the project (faster for private work):

1. On your laptop, zip the `MeMo/` folder (excluding `.venv/` and `data/` if you'll upload that separately).
2. In Colab's left sidebar, click the folder icon and upload the zip.
3. Run:

In [4]:
!unzip -q MeMo.zip -d memo_project
%cd memo_project

unzip:  cannot find or open MeMo.zip, MeMo.zip.zip or MeMo.zip.ZIP.
[Errno 2] No such file or directory: 'memo_project'
/content


## 3. Install Step B dependencies

Colab already has a CUDA torch installed — we use editable install with the `train` extra,
which is compatible with the pre-installed torch (won't reinstall it unless versions clash).

In [ ]:
!pip install -q -e ".[train]"

## 4. Upload the Step A dataset

Generate `data/reflection_qa.jsonl` locally (Step A), then upload it.

Option 1 — drag-and-drop into Colab's `memo_project/data/` folder.

Option 2 — use this cell to pick the file from your laptop:

In [ ]:
import os
os.makedirs("data", exist_ok=True)

from google.colab import files  # type: ignore
uploaded = files.upload()  # browse for reflection_qa.jsonl
for fname in uploaded:
    if fname != "data/reflection_qa.jsonl":
        os.rename(fname, "data/reflection_qa.jsonl")
        break
!ls -la data/

## 5. Inspect the dataset

In [ ]:
!memo-step-b inspect --data data/reflection_qa.jsonl --tokenizer-check --sample 5

Expect:
- `pairs: N` where N matches what Step A produced.
- `avg answer-token fraction` somewhere between 5% and 40% (higher = longer answers, lower = short ones — both are fine).
- The printed sample shows a Qwen chat template wrapping your Q&A.

## 6. Train the Memory Model

On a T4 with the 1.5B base model + LoRA + the tiny sample corpus (~50 pairs), this completes in
a few minutes. For real corpora, scale `--epochs` / dataset size.

If you hit OOM, drop `--max-length` to 1024 or use `--qlora` (requires the qlora extra).

In [ ]:
!memo-step-b train \
    --data data/reflection_qa.jsonl \
    --output memory_model_ckpt \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --epochs 3 \
    --batch-size 1 \
    --grad-accum 16 \
    --max-length 1024

## 7. Quick inference probe

Compare the trained model's answer to what the base model would say. The trained model should
answer from the corpus; the base model will hallucinate or refuse.

In [ ]:
!memo-step-b infer --adapter memory_model_ckpt --question "Who was Dr. Elena Marchetti and where did she work?"

In [ ]:
!memo-step-b infer --adapter memory_model_ckpt --question "What award did Prof. James Holbrook win and in what year?"

## 8. (Optional) Save the adapter to Google Drive

Colab disks are ephemeral. Mount Drive and copy the checkpoint to keep it.

In [ ]:
from google.colab import drive  # type: ignore
drive.mount("/content/drive")
!cp -r memory_model_ckpt /content/drive/MyDrive/memo_memory_model_ckpt
!ls /content/drive/MyDrive/memo_memory_model_ckpt